In [42]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [48]:
raw = spark.read.csv("hdfs://namenode:9000/data/raw/accidentes/", header=True, inferSchema=True)
raw.write.mode("overwrite").parquet("hdfs://namenode:9000/data/staging/accidentes")

In [53]:
raw.count()

7728394

In [49]:
stg = spark.read.parquet("hdfs://namenode:9000/data/staging/accidentes/")

In [54]:
stg.count()

7728394

In [27]:
stg.columns

['ID',
 'Source',
 'Severity',
 'Start_Time',
 'End_Time',
 'Start_Lat',
 'Start_Lng',
 'End_Lat',
 'End_Lng',
 'Distance(mi)',
 'Description',
 'Street',
 'City',
 'County',
 'State',
 'Zipcode',
 'Country',
 'Timezone',
 'Airport_Code',
 'Weather_Timestamp',
 'Temperature(F)',
 'Wind_Chill(F)',
 'Humidity(%)',
 'Pressure(in)',
 'Visibility(mi)',
 'Wind_Direction',
 'Wind_Speed(mph)',
 'Precipitation(in)',
 'Weather_Condition',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Turning_Loop',
 'Sunrise_Sunset',
 'Civil_Twilight',
 'Nautical_Twilight',
 'Astronomical_Twilight']

In [44]:
accidents = stg.select(stg.columns[0:19])

In [45]:
accidents.show(2)

+---------+-------+--------+-------------------+-------------------+---------+-----------+---------+-----------+------------+--------------------+---------+-----------+---------+-----+----------+-------+----------+------------+
|       ID| Source|Severity|         Start_Time|           End_Time|Start_Lat|  Start_Lng|  End_Lat|    End_Lng|Distance(mi)|         Description|   Street|       City|   County|State|   Zipcode|Country|  Timezone|Airport_Code|
+---------+-------+--------+-------------------+-------------------+---------+-----------+---------+-----------+------------+--------------------+---------+-----------+---------+-----+----------+-------+----------+------------+
|A-5199312|Source1|       2|2022-10-07 23:27:00|2022-10-08 03:56:30|35.385605|-118.955944|35.386096|-118.954941|       0.066|Incident on KENT ...| Scott Pl|Bakersfield|     Kern|   CA|93306-3835|     US|US/Pacific|        KBFL|
|A-5199313|Source1|       4|2023-01-28 03:12:30|2023-01-28 05:22:43|37.717731| -80.04920

In [46]:
weather = stg.select("ID",*stg.columns[19:30])

weather.show(2)

+---------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+--------------------+-------+
|       ID|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|   Weather_Condition|Amenity|
+---------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+--------------------+-------+
|A-5199312|2022-10-07 23:54:00|          79.0|         79.0|       34.0|        29.5|           8.0|             E|            5.0|              0.0|                Fair|  false|
|A-5199313|2023-01-28 03:15:00|          32.0|         19.0|       64.0|       26.21|          10.0|             W|           23.0|              0.0|Partly Cloudy / W...|  false|
+---------+-------------------+--------------+-------------+-----------+------------+--------------+-----

In [47]:
from pyspark.sql import functions as F

cols = [c for c in accidents.columns if c != "ID"]

nulos_por_id = (
    accidents.select(
        "ID",
        F.array_compact(
            F.array(*[F.when(F.col(f"`{c}`").isNull(), F.lit(c)) for c in cols])
        ).alias("columnas_nulas"),
    )
    .withColumn("n_nulos", F.size("columnas_nulas"))
    .filter("n_nulos > 0")
)

nulos_por_id.orderBy(F.desc("n_nulos")).show(10, truncate=False)

+---------+---------------------------------------------------+-------+
|ID       |columnas_nulas                                     |n_nulos|
+---------+---------------------------------------------------+-------+
|A-3253821|[End_Lat, End_Lng, Zipcode, Timezone, Airport_Code]|5      |
|A-7795   |[End_Lat, End_Lng, Zipcode, Timezone, Airport_Code]|5      |
|A-1101119|[End_Lat, End_Lng, Zipcode, Timezone, Airport_Code]|5      |
|A-119882 |[End_Lat, End_Lng, Zipcode, Timezone, Airport_Code]|5      |
|A-760145 |[End_Lat, End_Lng, Zipcode, Timezone, Airport_Code]|5      |
|A-13890  |[End_Lat, End_Lng, Zipcode, Timezone, Airport_Code]|5      |
|A-1127154|[End_Lat, End_Lng, Zipcode, Timezone, Airport_Code]|5      |
|A-14516  |[End_Lat, End_Lng, Zipcode, Timezone, Airport_Code]|5      |
|A-3283195|[End_Lat, End_Lng, Zipcode, Timezone, Airport_Code]|5      |
|A-16209  |[End_Lat, End_Lng, Zipcode, Timezone, Airport_Code]|5      |
+---------+---------------------------------------------------+-